# Autocorrelation Analysis of Factual Errors

For each atomic fact, we encode:

- `0`: supported fact
- `1`: unsupported fact

Then we estimate the mean autocorrelation by lag and bootstrap confidence intervals across responses.

## How to Use

1. Install the analysis dependencies if needed: `pip install numpy matplotlib statsmodels`.
2. Update `DATASETS` if your final-decision JSONL files are in different locations.
3. Run the helper-function cells.
4. Run the summary, bootstrap, and plotting cells.

Each input file should be a JSONL file produced by `get_final_decisions.py`, with one row per entity and a `decisions` list containing `is_supported` labels.

## Configuration

Keep `N_BOOTSTRAP` high for final figures and lower it for quick checks.

In [ ]:
DATASETS = {
    "GPT-4o": "", # e.g. "../output/gpt-4o/gpt-4o_0_183_final_decisions.jsonl 
}

NLAGS = 8
N_BOOTSTRAP = 2000
RANDOM_SEED = 0


## Loading Helpers

In [ ]:
def notebook_dir() -> Path:
    """Return the expected directory for this notebook's relative paths."""
    cwd = Path.cwd()
    if cwd.name == "analysis" and cwd.parent.name == "gen_and_verify":
        return cwd
    repo_relative = cwd / "gen_and_verify" / "analysis"
    if repo_relative.exists():
        return repo_relative
    return cwd


def resolve_path(path: str | Path, base_dir: Path | None = None) -> Path:
    path = Path(path).expanduser()
    if path.is_absolute():
        return path
    return (base_dir or notebook_dir()) / path


def load_jsonl(path: str | Path, base_dir: Path | None = None) -> list[dict[str, Any]]:
    resolved = resolve_path(path, base_dir)
    with resolved.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_datasets(
    datasets: dict[str, str | Path],
    base_dir: Path | None = None,
    *,
    skip_missing: bool = True,
) -> dict[str, list[dict[str, Any]]]:
    loaded = {}
    for name, path in datasets.items():
        resolved = resolve_path(path, base_dir)
        if not resolved.exists():
            message = f"Missing file for {name}: {resolved}"
            if skip_missing:
                print(f"[skip] {message}")
                continue
            raise FileNotFoundError(message)
        loaded[name] = load_jsonl(resolved)
        print(f"[load] {name}: {len(loaded[name])} rows from {resolved}")
    return loaded


## Dataset Summaries

These functions give the basic descriptive statistics: factual precision, supported facts, total facts, and average response length.

In [ ]:
TRUE_VALUES = {True, "true", "True", "TRUE", "supported", "Supported", "S", "1", 1}


def is_supported_label(value: Any) -> bool:
    return value in TRUE_VALUES


def decisions_for(row: dict[str, Any]) -> list[dict[str, Any]]:
    return row.get("decisions", []) or []


def count_words(text: str) -> int:
    return len((text or "").split())


def factual_precision(row: dict[str, Any]) -> float:
    decisions = decisions_for(row)
    if not decisions:
        return np.nan
    supported = sum(is_supported_label(decision.get("is_supported")) for decision in decisions)
    return supported / len(decisions)


def dataset_summary(rows: list[dict[str, Any]]) -> dict[str, float]:
    fact_counts = np.array([len(decisions_for(row)) for row in rows], dtype=float)
    supported_counts = np.array([
        sum(is_supported_label(decision.get("is_supported")) for decision in decisions_for(row))
        for row in rows
    ], dtype=float)
    precisions = np.divide(
        supported_counts,
        fact_counts,
        out=np.full_like(supported_counts, np.nan),
        where=fact_counts > 0,
    )
    word_counts = np.array([count_words(row.get("vanilla_output", "")) for row in rows], dtype=float)
    return {
        "n_rows": float(len(rows)),
        "mean_factual_precision": float(np.nanmean(precisions)),
        "mean_supported_facts": float(np.nanmean(supported_counts)),
        "mean_all_facts": float(np.nanmean(fact_counts)),
        "mean_words": float(np.nanmean(word_counts)),
    }


def print_summary_table(summaries: dict[str, dict[str, float]]) -> None:
    header = f"{'Dataset':<24} {'Rows':>6} {'FP':>8} {'Supp.':>8} {'Facts':>8} {'Words':>8}"
    print(header)
    print("-" * len(header))
    for name, summary in summaries.items():
        print(
            f"{name:<24} "
            f"{summary['n_rows']:>6.0f} "
            f"{summary['mean_factual_precision']:>8.4f} "
            f"{summary['mean_supported_facts']:>8.2f} "
            f"{summary['mean_all_facts']:>8.2f} "
            f"{summary['mean_words']:>8.2f}"
        )


## Error-Series Construction

Each response becomes one binary sequence in generation order. Homogeneous sequences are kept here and skipped only when computing ACF, because ACF is undefined when variance is zero.

In [ ]:
def decision_to_error(decision: dict[str, Any]) -> int:
    return 0 if is_supported_label(decision.get("is_supported")) else 1


def build_error_series(
    rows: list[dict[str, Any]],
    *,
    only_verifiable: bool | None = None,
) -> list[list[int]]:
    series_list = []
    for row in rows:
        decisions = decisions_for(row)
        if only_verifiable is not None:
            expected = "Yes" if only_verifiable else "No"
            decisions = [d for d in decisions if d.get("verifiable") == expected]
        series = [decision_to_error(decision) for decision in decisions]
        if series:
            series_list.append(series)
    return series_list


def describe_error_series(series_list: list[list[int]]) -> dict[str, float]:
    lengths = np.array([len(series) for series in series_list], dtype=float)
    error_rates = np.array([np.mean(series) for series in series_list], dtype=float)
    return {
        "n_series": float(len(series_list)),
        "min_length": float(np.min(lengths)) if len(lengths) else np.nan,
        "max_length": float(np.max(lengths)) if len(lengths) else np.nan,
        "mean_length": float(np.mean(lengths)) if len(lengths) else np.nan,
        "mean_error_rate": float(np.mean(error_rates)) if len(error_rates) else np.nan,
    }


## Autocorrelation and Bootstrap Confidence Intervals

We skip sequences that are too short for the requested lag or have no variance. Bootstrap resamples responses with replacement and recomputes the mean ACF.

In [ ]:
def is_homogeneous(series: list[int]) -> bool:
    return len(series) > 0 and all(value == series[0] for value in series)


def valid_acf_series(series_list: list[list[int]], nlags: int) -> list[list[int]]:
    return [
        series
        for series in series_list
        if len(series) > nlags and not is_homogeneous(series)
    ]


def mean_acf_per_lag(series_list: list[list[int]], nlags: int) -> np.ndarray:
    usable = valid_acf_series(series_list, nlags)
    if not usable:
        raise ValueError("No valid error series remain after length and variance filtering.")
    acf_values = [acf(series, nlags=nlags, fft=False)[: nlags + 1] for series in usable]
    return np.mean(np.vstack(acf_values), axis=0)


def bootstrap_mean_acf(
    series_list: list[list[int]],
    *,
    nlags: int = 8,
    n_bootstrap: int = 2000,
    seed: int = 0,
) -> dict[str, np.ndarray | int]:
    usable = valid_acf_series(series_list, nlags)
    if not usable:
        raise ValueError("No valid error series remain after length and variance filtering.")

    rng = np.random.default_rng(seed)
    bootstrap_means = []
    for _ in range(n_bootstrap):
        sample_indices = rng.integers(0, len(usable), size=len(usable))
        sample = [usable[index] for index in sample_indices]
        bootstrap_means.append(mean_acf_per_lag(sample, nlags))

    bootstrap_means = np.vstack(bootstrap_means)
    return {
        "all_bootstrap_means": bootstrap_means,
        "mean_per_lag": np.mean(bootstrap_means, axis=0),
        "conf_interval": np.percentile(bootstrap_means, [2.5, 97.5], axis=0),
        "n_usable_series": len(usable),
    }


def print_acf_table(result: dict[str, np.ndarray | int], *, digits: int = 3) -> None:
    means = result["mean_per_lag"]
    conf_interval = result["conf_interval"]
    print(f"Usable series: {result['n_usable_series']}")
    for lag, mean_value in enumerate(means):
        lower = conf_interval[0][lag]
        upper = conf_interval[1][lag]
        print(f"lag {lag}: {mean_value:.{digits}f}, CI [{lower:.{digits}f}, {upper:.{digits}f}]")


## Plotting Helpers

In [ ]:
def plot_acf_with_conf_interval(
    result: dict[str, np.ndarray | int],
    *,
    label: str,
    title: str = "Autocorrelation of Factual Errors at Different Lags",
    color: tuple[float, float, float] = (130 / 255, 115 / 255, 180 / 255),
    ax: plt.Axes | None = None,
) -> plt.Axes:
    if ax is None:
        _, ax = plt.subplots(figsize=(6.4, 4.8), dpi=200)

    mean_values = result["mean_per_lag"]
    conf_interval = result["conf_interval"]
    lags = np.arange(len(mean_values))

    ax.plot(lags, mean_values, marker="o", linewidth=2.5, markersize=6, color=color, label=label)
    ax.fill_between(lags, conf_interval[0], conf_interval[1], color=color, alpha=0.18)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("Lag ($k$)")
    ax.set_ylabel("Autocorrelation Coefficient ($r_k$)")
    ax.set_title(title)
    ax.legend()
    return ax


def plot_acf_comparison(
    results: dict[str, dict[str, np.ndarray | int]],
    *,
    title: str = "Autocorrelation of Factual Errors by Model",
) -> plt.Axes:
    _, ax = plt.subplots(figsize=(7.2, 4.8), dpi=200)
    for label, result in results.items():
        mean_values = result["mean_per_lag"]
        lags = np.arange(len(mean_values))
        ax.plot(lags, mean_values, marker="o", linewidth=2, markersize=5, label=label)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("Lag ($k$)")
    ax.set_ylabel("Autocorrelation Coefficient ($r_k$)")
    ax.set_title(title)
    ax.legend()
    return ax


## Load Data and Print Basic Summaries

If a file is missing, the loader skips it and prints the missing path. Set `skip_missing=False` in `load_datasets` if you prefer a hard failure.

In [ ]:
datasets = load_datasets(DATASETS, skip_missing=True)
summaries = {name: dataset_summary(rows) for name, rows in datasets.items()}
print_summary_table(summaries)


## Build Error Series

The printed diagnostics help catch unexpectedly short outputs or datasets where every response is homogeneous.

In [ ]:
error_series = {name: build_error_series(rows) for name, rows in datasets.items()}

for name, series_list in error_series.items():
    print(name, describe_error_series(series_list))


## Run Bootstrap ACF

In [ ]:
acf_results = {
    name: bootstrap_mean_acf(
        series_list,
        nlags=NLAGS,
        n_bootstrap=N_BOOTSTRAP,
        seed=RANDOM_SEED,
    )
    for name, series_list in error_series.items()
}

for name, result in acf_results.items():
    print(f"\n{name}")
    print_acf_table(result)


## Plot One Dataset

Change `dataset_name` to inspect one model with its bootstrap confidence interval.

In [ ]:
dataset_name = "GPT-4o"

if dataset_name in acf_results:
    plot_acf_with_conf_interval(acf_results[dataset_name], label=dataset_name)
    plt.show()
else:
    print(f"{dataset_name!r} is not loaded. Available datasets: {list(acf_results)}")
